# BERT-Based Proposed Version Simulation 2 — Binary Spam vs Ham

Stacking ensemble of LoRA-fine-tuned multilingual transformers (mBERT, XLM-R, MuRIL, DistilmBERT)
on the merged English + Bangla + Code-mixed SMS dataset.

**Binary task:** `ham = 0`, `spam = 1`. No source column.

In [ ]:
ReportFolderName = 'BERT-Based_Proposed_Version_Simulation2'

In [ ]:
import shutil
import os

# Keywords for folders to delete
folders_to_delete = ["logs", ReportFolderName, "results", "sample_data", "adapters"]

for item in os.listdir("."):
    if os.path.isdir(item) and any(keyword in item for keyword in folders_to_delete):
        shutil.rmtree(item)
        print(f"✅ Deleted folder: {item}")

print("\n🎯 Cleanup completed.")

✅ Deleted folder: sample_data

🎯 Cleanup completed.


In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"

!pip install -q --upgrade transformers datasets peft accelerate scikit-learn tqdm "torchao>=0.16.0"

import torch, string, numpy as np, pandas as pd
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments
)
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.2 MB/s eta 0:00:00


In [ ]:
random_state = 44

import os, random
import numpy as np
import torch

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(random_state)

In [ ]:
import re

def pre_process_sms(text):
    text = re.sub(r"http\S+", "URL", text)
    # Bangladeshi mobile numbers (optional +88 country code, optional separators)
    text = re.sub(r'(\+?88)?[\s-]?01[3-9][\s-]?\d{4}[\s-]?\d{4}', 'PHONE', text)
    # Generic phone-like number sequences (7+ digits, optional separators)
    text = re.sub(r'(?<!\d)(\+?\d[\d\s-]{6,}\d)(?!\d)', 'PHONE', text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.strip()

def preprocess_text(batch):
    batch["text"] = pre_process_sms(batch["text"])
    return batch

In [ ]:
# Binary label scheme: ham vs spam
label2id = {"ham": 0, "spam": 1}
id2label = {v: k for k, v in label2id.items()}

def encode_labels(batch):
    batch["label"] = label2id[batch["label"]]
    return batch

In [ ]:
# ============================================================
# LOAD & MERGE THE THREE DATASETS  (binary: spam vs ham, no source)
#   Downloaded from the HF Hub dataset repo:
#     https://huggingface.co/datasets/shariul-islam/simulation2_dataset
#   spam.csv             = TAB-delimited  (English / UCI SMS Spam Collection)
#   bangla_spam_sms.csv  = COMMA-delimited (Bangla)
#   code_mixed_sms.csv   = COMMA-delimited (Bangla-English code-mixed)
#   All headerless: label in column 0, text in column 1.
# ============================================================
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "shariul-islam/simulation2_dataset"

def _hf_csv(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                           repo_type="dataset")

spam_path = _hf_csv("spam.csv")
bn_path   = _hf_csv("bangla_spam_sms.csv")
cm_path   = _hf_csv("code_mixed_sms.csv")

en = pd.read_csv(spam_path, sep="\t", header=None,
                 names=["label", "text"], encoding="latin-1",
                 quoting=3)   # QUOTE_NONE
bn = pd.read_csv(bn_path, header=None,
                 names=["label", "text"], encoding="utf-8")
cm = pd.read_csv(cm_path, header=None,
                 names=["label", "text"], encoding="utf-8")

def norm_label(x):
    return "spam" if str(x).strip().lower() == "spam" else "ham"

frames = []
for d in (en, bn, cm):
    d = d.copy()
    d["label"] = d["label"].map(norm_label)
    d["text"]  = d["text"].astype(str)
    frames.append(d[["text", "label"]])

df = (pd.concat(frames, ignore_index=True)
        .dropna()
        .drop_duplicates("text")
        .reset_index(drop=True))

print("Merged dataset:", df.shape)
print(df["label"].value_counts())
print("\nPer-source counts:")
for name, d in [("English", en), ("Bangla", bn), ("Code-mixed", cm)]:
    print(f"  {name:11s}: {len(d):5d}")

spam.csv:   0%|          | 0.00/478k [00:00<?, ?B/s]

bangla_spam_sms.csv:   0%|          | 0.00/112k [00:00<?, ?B/s]

code_mixed_sms.csv:   0%|          | 0.00/78.4k [00:00<?, ?B/s]

Merged dataset: (6150, 2)
label
ham     4961
spam    1189
Name: count, dtype: int64

Per-source counts:
  English    :  5574
  Bangla     :   504
  Code-mixed :   500


In [ ]:
# ============================================================
# Stratified 80 / 10 / 10 split  ->  train / validation / test
# ============================================================
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=random_state, stratify=df["label"])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=random_state, stratify=temp_df["label"])

dataset = DatasetDict({
    "train":      Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test":       Dataset.from_pandas(test_df.reset_index(drop=True)),
})
print({k: len(v) for k, v in dataset.items()})

{'train': 4920, 'validation': 615, 'test': 615}


In [ ]:
dataset = dataset.map(encode_labels)
dataset = dataset.map(preprocess_text)

Map:   0%|          | 0/4920 [00:00<?, ? examples/s]

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Map:   0%|          | 0/4920 [00:00<?, ? examples/s]

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

Map:   0%|          | 0/615 [00:00<?, ? examples/s]

In [ ]:
dataset['train'][2]

{'text': 'fun fact although you would think armand would eventually build up a tolerance or some shit considering how much he smokes he gets fucked up in like 2 hits',
 'label': 0}

In [ ]:
train_dataset = dataset['train']
val_dataset   = dataset['validation']
test_dataset  = dataset['test']

In [ ]:
print("Train label balance:")
print(pd.DataFrame(train_dataset)["label"].value_counts())
print("\nTest label balance:")
print(pd.DataFrame(test_dataset)["label"].value_counts())

Train label balance:
label
0    3969
1     951
Name: count, dtype: int64

Test label balance:
label
0    496
1    119
Name: count, dtype: int64


In [ ]:
def save_model_into_huggingface(model, tokenizer, model_alias):
    # ── Save LoRA adapter to Hugging Face Hub ──
    from huggingface_hub import HfApi, create_repo

    HF_USERNAME = "shariul-islam"   # your HF username
    repo_id = f"{HF_USERNAME}/proposed-binary-{model_alias.lower().replace('/', '-')}"

    try:
        create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
        print(f"✅ Repo ready: {repo_id}")
    except Exception as e:
        print(f"Repo note: {e}")

    adapter_local_path = f"./adapters/{model_alias}"
    model.save_pretrained(adapter_local_path)
    tokenizer.save_pretrained(adapter_local_path)

    model.push_to_hub(repo_id, commit_message=f"Add LoRA adapter: {model_alias}")
    tokenizer.push_to_hub(repo_id, commit_message=f"Add tokenizer: {model_alias}")
    print(f"✅ Pushed to HF: https://huggingface.co/{repo_id}")

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    accuracy_score, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize


def evaluate_and_report(trainer, test_dataset, label2id, model_alias, include_source=False):
    """
    Generates and saves evaluation reports for a trained model:
      - Classification report (text, CSV)
      - Confusion matrix
      - ROC curve (binary, positive class = spam)
      - Appends model summary to the report folder
    """
    print(f"\n📊 Generating Evaluation Report for {model_alias}")

    id2label = {v: k for k, v in label2id.items()}
    report_dir = f"./{ReportFolderName}"
    os.makedirs(report_dir, exist_ok=True)

    # 1️⃣ Predictions
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_dataset["label"]

    np.save(f"{report_dir}/{model_alias}_y_true.npy", y_true)
    np.save(f"{report_dir}/{model_alias}_y_pred.npy", y_pred)

    # Prediction CSV: SMS_Text, True_Label, Predicted_Label, Is_Correct
    sms_text = test_dataset["text"]
    true_lbl = [id2label[int(t)] for t in y_true]
    pred_lbl = [id2label[int(p)] for p in y_pred]
    is_correct = [int(t == p) for t, p in zip(y_true, y_pred)]

    pd.DataFrame({
        "SMS_Text":        sms_text,
        "True_Label":      true_lbl,
        "Predicted_Label": pred_lbl,
        "Is_Correct":      is_correct,
    }).to_csv(
        f"{report_dir}/{model_alias}_proposed_version_predictions.csv",
        index=False, encoding="utf-8-sig")

    class_names = list(label2id.keys())

    # 2️⃣ Classification report
    report_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    overall_report_dict = classification_report(
        y_true, y_pred, target_names=class_names, output_dict=True)

    with open(f"{report_dir}/{model_alias}_proposed_version_classification_report.txt", "w") as f:
        f.write(report_text)
    pd.DataFrame(overall_report_dict).transpose().round(4).to_csv(
        f"{report_dir}/{model_alias}_proposed_version_classification_report.csv")
    print(report_text)

    # 3️⃣ Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(label2id.values()))
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"Confusion Matrix - {model_alias}")
    plt.tight_layout()
    plt.savefig(f"{report_dir}/{model_alias}_proposed_version_confusion_matrix.png")
    plt.close()

    # 4️⃣ ROC curve (binary, positive class = spam)
    try:
        y_score = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()
        pos = label2id["spam"]
        fpr, tpr, _ = roc_curve(np.array(y_true) == pos, y_score[:, pos])
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, lw=2, label=f"spam (AUC = {roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], "k--", label="Random")
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve - {model_alias}")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(f"{report_dir}/{model_alias}_proposed_version_roc_curve.png")
        plt.close()
    except Exception as e:
        print(f"⚠️ Skipping ROC curve for {model_alias}: {e}")

    # 5️⃣ Summary CSV (append)
    acc       = round(overall_report_dict["accuracy"], 4)
    precision = round(overall_report_dict["weighted avg"]["precision"], 4)
    recall    = round(overall_report_dict["weighted avg"]["recall"], 4)
    f1        = round(overall_report_dict["weighted avg"]["f1-score"], 4)

    summary_dict = {"model": model_alias, "accuracy": acc,
                    "precision": precision, "recall": recall, "f1": f1}

    summary_path = os.path.join(report_dir, "BERT_proposed_version_summary.csv")
    if os.path.exists(summary_path):
        existing = pd.read_csv(summary_path)
        existing = pd.concat([existing, pd.DataFrame([summary_dict])], ignore_index=True)
        existing.to_csv(summary_path, index=False)
    else:
        pd.DataFrame([summary_dict]).to_csv(summary_path, index=False)

    print(f"\n✅ {model_alias} → Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print(f"✅ All reports saved in {report_dir}")
    return summary_dict

In [ ]:
def tokenize(batch):
    # Dynamic padding handled by DataCollatorWithPadding -> no padding here
    tokenized = tokenizer(batch["text"], truncation=True, max_length=128)
    tokenized["label"] = batch["label"]
    return tokenized


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
reports_dir = f"./{ReportFolderName}"
os.makedirs(reports_dir, exist_ok=True)

In [ ]:
# ============================================
# DEFINE BASE MODELS
# ============================================
base_models = {
    "mBERT":        "bert-base-multilingual-cased",
    "XLM-RoBERTa":  "xlm-roberta-base",
    "Muril":        "google/muril-large-cased",
    "Distil-mBERT": "distilbert-base-multilingual-cased",
}

meta_train_features = []
meta_test_features  = []
all_model_results   = []

# Set to False for a pure local test run (skips Hugging Face Hub upload)
PUSH_TO_HUB = False

# ============================================
# LOOP THROUGH EACH BASE MODEL
# ============================================
for model_alias, model_name in base_models.items():
    print(f"\n🔥 Fine-tuning Base Model: {model_alias}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Tokenize into LOCAL variables so re-running with a different model
    # does NOT reuse a previous tokenizer's columns. Global *_dataset stays raw.
    train_tok = train_dataset.map(tokenize, batched=True)
    val_tok   = val_dataset.map(tokenize, batched=True)
    test_tok  = test_dataset.map(tokenize, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )

    target_modules = ["query", "key", "value", "dense"]
    if model_name == "distilbert-base-multilingual-cased":
        target_modules = ["attention.q_lin", "attention.k_lin",
                          "attention.v_lin", "attention.out_lin"]  # DistilBERT names

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS",
    )
    print(target_modules)
    model = get_peft_model(base_model, lora_config)

    training_args = TrainingArguments(
        output_dir=f"./results_{model_alias}",
        learning_rate=3e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=10,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        fp16=True,
        seed=random_state,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    if PUSH_TO_HUB:
        save_model_into_huggingface(model, tokenizer, model_alias)

    # Collect softmax probabilities for stacking
    preds_train = trainer.predict(train_tok)
    preds_test  = trainer.predict(test_tok)

    meta_train_features.append(
        torch.softmax(torch.tensor(preds_train.predictions), dim=1).numpy())
    meta_test_features.append(
        torch.softmax(torch.tensor(preds_test.predictions), dim=1).numpy())

    # Evaluate model and save reports (no source in binary setup)
    metrics = evaluate_and_report(trainer, test_tok, label2id, model_alias, include_source=False)
    all_model_results.append(metrics)


🔥 Fine-tuning Base Model: mBERT


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


['attention.q_lin', 'attention.k_lin', 'attention.v_lin', 'attention.out_lin']


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.128530,0.952846,0.952266,0.952846,0.952458
2,No log,0.103543,0.969106,0.968790,0.969106,0.968746
3,No log,0.095075,0.972358,0.972108,0.972358,0.972130
4,0.165538,0.085965,0.973984,0.973766,0.973984,0.973726
5,0.165538,0.081443,0.973984,0.973766,0.973984,0.973726
6,0.165538,0.081258,0.973984,0.973825,0.973984,0.973635
7,0.082047,0.074538,0.973984,0.973774,0.973984,0.973814
8,0.082047,0.075376,0.973984,0.973825,0.973984,0.973635
9,0.082047,0.078232,0.973984,0.973825,0.973984,0.973635
10,0.073762,0.077067,0.973984,0.973825,0.973984,0.973635



📊 Generating Evaluation Report for Distil-mBERT


              precision    recall  f1-score   support

         ham     0.9859    0.9859    0.9859       496
        spam     0.9412    0.9412    0.9412       119

    accuracy                         0.9772       615
   macro avg     0.9635    0.9635    0.9635       615
weighted avg     0.9772    0.9772    0.9772       615


✅ Distil-mBERT → Accuracy: 0.9772, F1: 0.9772
✅ All reports saved in ./BERT-Based_Proposed_Binary_Version


In [ ]:
# ============================================
# STACKING META-LEARNER
# Stack the per-model softmax probabilities and train a
# Logistic Regression meta-classifier (binary spam vs ham).
# ============================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Concatenate base-model probabilities horizontally -> meta feature matrix
X_meta_train = np.hstack(meta_train_features)
X_meta_test  = np.hstack(meta_test_features)

y_meta_train = np.array(train_dataset["label"])
y_meta_test  = np.array(test_dataset["label"])

print("Meta-train features:", X_meta_train.shape)
print("Meta-test  features:", X_meta_test.shape)

meta_clf = LogisticRegression(max_iter=1000, random_state=random_state)
meta_clf.fit(X_meta_train, y_meta_train)

y_meta_pred = meta_clf.predict(X_meta_test)

class_names = list(label2id.keys())
print("\n🏆 STACKING ENSEMBLE RESULTS")
print(classification_report(y_meta_test, y_meta_pred, target_names=class_names, digits=4))

# Save stacking report + confusion matrix
import matplotlib.pyplot as plt, seaborn as sns
report_dir = f"./{ReportFolderName}"

stack_report = classification_report(
    y_meta_test, y_meta_pred, target_names=class_names, output_dict=True)
pd.DataFrame(stack_report).transpose().round(4).to_csv(
    f"{report_dir}/STACKING_ensemble_classification_report.csv")

cm = confusion_matrix(y_meta_test, y_meta_pred, labels=list(label2id.values()))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix - Stacking Ensemble")
plt.tight_layout()
plt.savefig(f"{report_dir}/STACKING_ensemble_confusion_matrix.png")
plt.close()

# Append stacking row to the summary
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
acc = round(accuracy_score(y_meta_test, y_meta_pred), 4)
p, r, f1, _ = precision_recall_fscore_support(y_meta_test, y_meta_pred, average="weighted")
stack_summary = {"model": "Stacking-Ensemble", "accuracy": acc,
                 "precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}

summary_path = f"{report_dir}/BERT_proposed_version_summary.csv"
if os.path.exists(summary_path):
    existing = pd.read_csv(summary_path)
    existing = pd.concat([existing, pd.DataFrame([stack_summary])], ignore_index=True)
    existing.to_csv(summary_path, index=False)
else:
    pd.DataFrame([stack_summary]).to_csv(summary_path, index=False)

print(f"\n✅ Stacking ensemble → Accuracy: {acc:.4f}, F1: {round(f1,4):.4f}")
print(pd.read_csv(summary_path))

Meta-train features: (4920, 8)
Meta-test  features: (615, 8)

🏆 STACKING ENSEMBLE RESULTS
              precision    recall  f1-score   support

         ham     0.9940    0.9960    0.9950       496
        spam     0.9831    0.9748    0.9789       119

    accuracy                         0.9919       615
   macro avg     0.9885    0.9854    0.9869       615
weighted avg     0.9919    0.9919    0.9919       615


✅ Stacking ensemble → Accuracy: 0.9919, F1: 0.9919
               model  accuracy  precision  recall      f1
0              mBERT    0.9789     0.9788  0.9789  0.9788
1        XLM-RoBERTa    0.9821     0.9824  0.9821  0.9822
2              Muril    0.9919     0.9919  0.9919  0.9919
3       Distil-mBERT    0.9772     0.9772  0.9772  0.9772
4  Stacking-Ensemble    0.9919     0.9919  0.9919  0.9919


In [ ]:
import shutil, os
try:
    from google.colab import files
    COLAB = True
except ImportError:
    COLAB = False

folders_to_download = [ReportFolderName]

for folder in folders_to_download:
    if os.path.exists(folder):
        zip_filename = f"{folder}.zip"
        shutil.make_archive(folder, 'zip', folder)
        print(f"✅ Created '{zip_filename}'")
        if COLAB:
            files.download(zip_filename)
    else:
        print(f"⚠️ Folder '{folder}' not found")

✅ Created 'BERT-Based_Proposed_Binary_Version.zip'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>